# 01 — Data Exploration

Load and explore SO 2025, SO 2023, and aijobs.net datasets. Filter to full-time employed, convert salaries to USD, save processed versions.

In [1]:
import os, warnings
warnings.filterwarnings("ignore")
# ensure kernel runs from repo root regardless of notebook location
os.chdir("/app")
import pathlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA_RAW = pathlib.Path("data/raw")
DATA_PROC = pathlib.Path("data/processed")
DATA_PROC.mkdir(parents=True, exist_ok=True)


## Stack Overflow Developer Survey 2025

In [2]:
so25 = pd.read_csv(DATA_RAW / "stackoverflow_2025.csv", low_memory=False)
print(f"Shape: {so25.shape}")
print(f"\nKey columns present: {[c for c in ['DevType','LanguageHaveWorkedWith','DatabaseHaveWorkedWith','WebframeHaveWorkedWith','WorkExp','EdLevel','Country','Employment','RemoteWork','CompTotal','Currency','JobSat','AISelect'] if c in so25.columns]}")
print(f"\nMissing values (key cols):")
key_cols = ['DevType','LanguageHaveWorkedWith','WorkExp','EdLevel','Country','Employment','RemoteWork','CompTotal','Currency','JobSat','AISelect']
for c in key_cols:
    if c in so25.columns:
        pct = 100 * so25[c].isna().mean()
        print(f"  {c}: {so25[c].isna().sum():,} ({pct:.1f}%)")


Shape: (49191, 172)

Key columns present: ['DevType', 'LanguageHaveWorkedWith', 'DatabaseHaveWorkedWith', 'WebframeHaveWorkedWith', 'WorkExp', 'EdLevel', 'Country', 'Employment', 'RemoteWork', 'CompTotal', 'Currency', 'JobSat', 'AISelect']

Missing values (key cols):
  DevType: 5,511 (11.2%)
  LanguageHaveWorkedWith: 17,520 (35.6%)
  WorkExp: 6,298 (12.8%)
  EdLevel: 1,042 (2.1%)
  Country: 13,754 (28.0%)
  Employment: 852 (1.7%)
  RemoteWork: 15,411 (31.3%)
  CompTotal: 24,325 (49.5%)
  Currency: 13,754 (28.0%)
  JobSat: 22,521 (45.8%)
  AISelect: 15,471 (31.5%)


In [3]:
print("Employment values:")
print(so25["Employment"].value_counts().to_string())
print("\nCurrency top 10:")
print(so25["Currency"].value_counts().head(10).to_string())
print("\nJobSat distribution:")
print(so25["JobSat"].value_counts().sort_index().to_string())


Employment values:
Employment
Employed                                                33750
Independent contractor, freelancer, or self-employed     6708
Student                                                  4428
Not employed                                             2227
Retired                                                   708
I prefer not to say                                       518

Currency top 10:
Currency
EUR European Euro           9618
USD United States dollar    8780
INR Indian rupee            2484
GBP Pound sterling          2047
CAD Canadian dollar         1283
PLN Polish zloty             826
AUD\tAustralian dollar       800
BRL Brazilian real           775
UAH Ukrainian hryvnia        622
SEK\tSwedish krona           601

JobSat distribution:
JobSat
0.0      242
1.0      165
2.0      439
3.0      811
4.0      803
5.0     1896
6.0     3230
7.0     5590
8.0     6972
9.0     3598
10.0    2924


In [4]:
# Filter to full-time employed only
so25_emp = so25[so25["Employment"] == "Employed"].copy()
print(f"Full-time employed: {len(so25_emp):,} rows (of {len(so25):,})")

# Currency → USD conversion (approximate 2025 exchange rates)
RATES = {
    "USD United States dollar": 1.000,
    "EUR European Euro": 1.080,
    "GBP Pound sterling": 1.270,
    "INR Indian rupee": 0.012,
    "CAD Canadian dollar": 0.740,
    "AUD Australian dollar": 0.650,
    "BRL Brazilian real": 0.190,
    "CHF Swiss franc": 1.130,
    "SEK Swedish krona": 0.095,
    "NOK Norwegian krone": 0.094,
    "DKK Danish krone": 0.145,
    "PLN Polish zloty": 0.250,
    "CZK Czech koruna": 0.044,
    "HUF Hungarian forint": 0.0028,
    "TRY Turkish lira": 0.030,
    "MXN Mexican peso": 0.058,
    "SGD Singapore dollar": 0.740,
    "ILS Israeli new shekel": 0.270,
    "UAH Ukrainian hryvnia": 0.024,
    "JPY Japanese yen": 0.0067,
    "KRW South Korean won": 0.00072,
    "ZAR South African rand": 0.055,
    "RON Romanian leu": 0.220,
    "BGN Bulgarian lev": 0.550,
}
so25_emp["rate"] = so25_emp["Currency"].map(RATES)
so25_emp["CompUSD"] = so25_emp["CompTotal"] * so25_emp["rate"]

# Remove missing salary, no recognized currency, and outliers
so25_clean = so25_emp[
    so25_emp["CompUSD"].notna() &
    so25_emp["WorkExp"].notna() &
    so25_emp["DevType"].notna() &
    (so25_emp["CompUSD"] >= 10_000) &
    (so25_emp["CompUSD"] <= 400_000)
].copy()

print(f"After cleaning: {len(so25_clean):,} rows with valid salary + experience + role")
print(f"  Median salary: ${so25_clean['CompUSD'].median():,.0f}")
print(f"  Mean salary:   ${so25_clean['CompUSD'].mean():,.0f}")
print(f"  Salary range:  ${so25_clean['CompUSD'].min():,.0f} – ${so25_clean['CompUSD'].max():,.0f}")


Full-time employed: 33,750 rows (of 49,191)
After cleaning: 13,739 rows with valid salary + experience + role
  Median salary: $84,970
  Mean salary:   $101,435
  Salary range:  $10,000 – $400,000


## Stack Overflow Developer Survey 2023

In [5]:
so23 = pd.read_csv(DATA_RAW / "stackoverflow_2023.csv", low_memory=False)
print(f"Shape: {so23.shape}")
print(f"\nKey columns:")
key23 = ["DevType","LanguageHaveWorkedWith","WorkExp","EdLevel","Country","Employment","RemoteWork","ConvertedCompYearly","AIBen","AISelect"]
for c in key23:
    if c in so23.columns:
        pct = 100 * so23[c].isna().mean()
        print(f"  {c}: {so23[c].isna().sum():,} missing ({pct:.1f}%)")
    else:
        print(f"  {c}: NOT PRESENT")


Shape: (89184, 84)

Key columns:
  DevType: 12,312 missing (13.8%)
  LanguageHaveWorkedWith: 2,044 missing (2.3%)
  WorkExp: 45,605 missing (51.1%)
  EdLevel: 1,211 missing (1.4%)
  Country: 1,211 missing (1.4%)
  Employment: 1,286 missing (1.4%)
  RemoteWork: 15,374 missing (17.2%)
  ConvertedCompYearly: 41,165 missing (46.2%)
  AIBen: 27,788 missing (31.2%)
  AISelect: 1,211 missing (1.4%)


In [6]:
so23_emp = so23[so23["Employment"].str.contains("Employed", na=False)].copy()
print(f"Full-time + contractor employed: {len(so23_emp):,} rows")

# SO 2023 already has ConvertedCompYearly (USD)
so23_emp = so23_emp.rename(columns={"ConvertedCompYearly": "CompUSD"})

so23_clean = so23_emp[
    so23_emp["CompUSD"].notna() &
    so23_emp["WorkExp"].notna() &
    so23_emp["DevType"].notna() &
    (so23_emp["CompUSD"] >= 10_000) &
    (so23_emp["CompUSD"] <= 400_000)
].copy()

print(f"After cleaning: {len(so23_clean):,} rows")
print(f"  Median salary: ${so23_clean['CompUSD'].median():,.0f}")
print(f"AIBen values:")
if "AIBen" in so23_clean.columns:
    print(so23_clean["AIBen"].value_counts().to_string())


Full-time + contractor employed: 65,418 rows
After cleaning: 28,648 rows
  Median salary: $78,069
AIBen values:
AIBen
Somewhat trust                7004
Neither trust nor distrust    6068
Somewhat distrust             4922
Highly distrust               1235
Highly trust                   362


## aijobs.net Salaries (2020–2025)

In [7]:
aijobs = pd.read_csv(DATA_RAW / "aijobs_salaries.csv")
print(f"Shape: {aijobs.shape}")
print(f"\nDtypes:\n{aijobs.dtypes.to_string()}")
print(f"\nMissing values:\n{aijobs.isna().sum().to_string()}")
print(f"\nWork years: {sorted(aijobs['work_year'].unique())}")
print(f"\nExperience levels: {aijobs['experience_level'].value_counts().to_string()}")
print(f"\nTop 15 job titles:")
print(aijobs["job_title"].value_counts().head(15).to_string())
print(f"\nMedian salary by year:")
print(aijobs.groupby("work_year")["salary_in_usd"].median().to_string())


Shape: (151445, 11)

Dtypes:
work_year              int64
experience_level      object
employment_type       object
job_title             object
salary                 int64
salary_currency       object
salary_in_usd          int64
employee_residence    object
remote_ratio           int64
company_location      object
company_size          object

Missing values:
work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0

Work years: [2020, 2021, 2022, 2023, 2024, 2025]

Experience levels: experience_level
SE    87491
MI    46128
EN    13663
EX     4163

Top 15 job titles:
job_title
Data Scientist               18751
Software Engineer            16948
Data Engineer                16352
Data Analyst                 13779
Engineer                     11004
Machine Learning Engineer     8

## Salary Distribution Visualizations

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(so25_clean["CompUSD"] / 1_000, bins=40, color="#4C72B0", edgecolor="white", linewidth=0.4)
axes[0].set_title("SO 2025 Salaries (USD, $K)")
axes[0].set_xlabel("Annual Salary ($K)")
axes[0].set_ylabel("Respondents")

axes[1].hist(so23_clean["CompUSD"] / 1_000, bins=40, color="#55A868", edgecolor="white", linewidth=0.4)
axes[1].set_title("SO 2023 Salaries (USD, $K)")
axes[1].set_xlabel("Annual Salary ($K)")

# aijobs: full-time only
ai_ft = aijobs[aijobs["employment_type"] == "FT"]
axes[2].hist(ai_ft["salary_in_usd"] / 1_000, bins=40, color="#DD8452", edgecolor="white", linewidth=0.4)
axes[2].set_title("aijobs.net Salaries (USD, $K)")
axes[2].set_xlabel("Annual Salary ($K)")

plt.tight_layout()
plt.savefig("figures/fig_data_exploration_salaries.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved salary distribution figure")


Saved salary distribution figure


In [9]:
# Top DevTypes with median salary (SO 2025)
so25_clean["DevType_primary"] = so25_clean["DevType"].str.split(";").str[0].str.strip()
role_salary = (
    so25_clean.groupby("DevType_primary")["CompUSD"]
    .agg(median="median", count="count")
    .query("count >= 20")
    .sort_values("median", ascending=True)
    .tail(15)
)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(role_salary.index, role_salary["median"] / 1_000, color="#4C72B0")
ax.set_xlabel("Median Salary ($K)")
ax.set_title("Median Salary by Role — SO 2025 (top 15 roles, ≥20 respondents)")
plt.tight_layout()
plt.savefig("figures/fig_salary_by_role_so25.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved role salary figure")


Saved role salary figure


## Save Processed Datasets

In [10]:
so25_clean.to_parquet(DATA_PROC / "so25_processed.parquet", index=False)
so23_clean.to_parquet(DATA_PROC / "so23_processed.parquet", index=False)
aijobs.to_parquet(DATA_PROC / "aijobs_processed.parquet", index=False)

print(f"Saved so25_processed.parquet  — {len(so25_clean):,} rows")
print(f"Saved so23_processed.parquet  — {len(so23_clean):,} rows")
print(f"Saved aijobs_processed.parquet — {len(aijobs):,} rows")


Saved so25_processed.parquet  — 13,739 rows
Saved so23_processed.parquet  — 28,648 rows
Saved aijobs_processed.parquet — 151,445 rows
